# Evaluate Adaptive vs. Simple Forecasts

This notebook provides a comprehensive evaluation of a micro-niche adaptive forecasting model, comparing its performance against two baseline models: a 3-point moving average and a naive (last-value) forecast. The evaluation is conducted on synthetic time series data, leveraging Mean Squared Error (MSE) and Mean Absolute Error (MAE) as key metrics.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

## Data Loading

We load the curated subset of synthetic time series and method outputs (`mini_demo_data.json`) from GitHub with a local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1560d8-micro-niche-adaptive-forecasting-for-sho/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=3) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
synthetic_series_data = data["synthetic_series_data"]
method_out_data = data["method_out_data"]
print(f"Loaded {len(synthetic_series_data)} synthetic time series examples.")

## Configuration

Define tunable parameters such as the moving average window size.

In [ ]:
# Config parameters
WINDOW_SIZE = 3

## Evaluation Metrics and Forecasting Functions

Define functions to compute MSE, MAE, naive forecasts, and moving average forecasts.

In [ ]:
def calculate_mse(actual, predictions):
    if not actual or not predictions or len(actual) != len(predictions):
        return None
    sum_sq_error = sum([(a - p) ** 2 for a, p in zip(actual, predictions)])
    return sum_sq_error / len(actual)

def calculate_mae(actual, predictions):
    if not actual or not predictions or len(actual) != len(predictions):
        return None
    sum_abs_error = sum([abs(a - p) for a, p in zip(actual, predictions)])
    return sum_abs_error / len(actual)

def naive_forecast(series, forecast_horizon):
    if not series:
        return []
    last_value = series[-1]
    return [last_value] * forecast_horizon

def moving_average_forecast(series, window_size, forecast_horizon):
    if len(series) < window_size:
        return [series[-1]] * forecast_horizon if series else []
    ma_value = sum(series[-window_size:]) / window_size
    return [ma_value] * forecast_horizon

## Run Evaluation Across Series

Compute metrics for naive, moving average, and adaptive forecasting models across all time series.

In [ ]:
all_results = []
overall_mse_naive = 0
overall_mae_naive = 0
overall_mse_ma = 0
overall_mae_ma = 0
overall_mse_adaptive = 0
overall_mae_adaptive = 0
total_forecasts = 0

for i, series in enumerate(synthetic_series_data):
    method_series_result = None
    for example in method_out_data['datasets'][0]['examples']:
        if example['metadata_series_id'] == i:
            method_series_result = example
            break
    
    if not method_series_result:
        continue

    actual_values = json.loads(method_series_result['output'])
    if not actual_values:
        continue

    forecast_horizon = len(actual_values)
    predictions_naive = naive_forecast(series[:-forecast_horizon], forecast_horizon)
    predictions_ma = moving_average_forecast(series[:-forecast_horizon], WINDOW_SIZE, forecast_horizon)
    predictions_adaptive = json.loads(method_series_result['predict_adaptive'])
    
    mse_naive = calculate_mse(actual_values, predictions_naive)
    mae_naive = calculate_mae(actual_values, predictions_naive)
    mse_ma = calculate_mse(actual_values, predictions_ma)
    mae_ma = calculate_mae(actual_values, predictions_ma)
    mse_adaptive = calculate_mse(actual_values, predictions_adaptive)
    mae_adaptive = calculate_mae(actual_values, predictions_adaptive)

    all_results.append({
        "metadata_series_id": i,
        "input": series,
        "output": actual_values,
        "predict_naive": predictions_naive,
        "predict_ma": predictions_ma,
        "predict_adaptive": predictions_adaptive,
        "eval_mse_naive": mse_naive,
        "eval_mae_naive": mae_naive,
        "eval_mse_ma": mse_ma,
        "eval_mae_ma": mae_ma,
        "eval_mse_adaptive": mse_adaptive,
        "eval_mae_adaptive": mae_adaptive
    })
    
    overall_mse_naive += mse_naive
    overall_mae_naive += mae_naive
    overall_mse_ma += mse_ma
    overall_mae_ma += mae_ma
    overall_mse_adaptive += mse_adaptive
    overall_mae_adaptive += mae_adaptive
    total_forecasts += 1

metrics_agg = {
    "avg_mse_naive": overall_mse_naive / total_forecasts,
    "avg_mae_naive": overall_mae_naive / total_forecasts,
    "avg_mse_ma": overall_mse_ma / total_forecasts,
    "avg_mae_ma": overall_mae_ma / total_forecasts,
    "avg_mse_adaptive": overall_mse_adaptive / total_forecasts,
    "avg_mae_adaptive": overall_mae_adaptive / total_forecasts
}

print("Aggregated Metrics:")
print(json.dumps(metrics_agg, indent=2))

## Results Summary & Visualization

Display aggregated performance metrics in a table and plot a comparison of forecast models against actual values for the series.

In [ ]:
# Table summary
table_data = [
    ["Naive Forecast", f"{metrics_agg['avg_mse_naive']:.4f}", f"{metrics_agg['avg_mae_naive']:.4f}"],
    ["Moving Average (3-pt)", f"{metrics_agg['avg_mse_ma']:.4f}", f"{metrics_agg['avg_mae_ma']:.4f}"],
    ["Adaptive Forecast", f"{metrics_agg['avg_mse_adaptive']:.4f}", f"{metrics_agg['avg_mae_adaptive']:.4f}"]
]
print(tabulate(table_data, headers=["Model", "Avg MSE", "Avg MAE"], tablefmt="fancy_grid"))

# Visualization
plt.figure(figsize=(10, 5))
res = all_results[0]
full_series = res["input"]
horizon = len(res["output"])
history_len = len(full_series) - horizon

x_history = list(range(history_len))
x_future = list(range(history_len, len(full_series)))

plt.plot(x_history, full_series[:history_len], 'bo-', label="History")
plt.plot(x_future, res["output"], 'ko--', label="Actual Future")
plt.plot(x_future, res["predict_naive"], 'r^:', label="Naive Forecast")
plt.plot(x_future, res["predict_ma"], 'gs:', label="Moving Average")
plt.plot(x_future, res["predict_adaptive"], 'm*-.', label="Adaptive Forecast", linewidth=2)

plt.title("Forecast Comparison on Sample Series #0")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()